# Quais temas cada bancada mais propõe

Análise temática dos **projetos de lei apresentados na Câmara dos Deputados na
57ª legislatura** (2023 até agosto de 2026), a partir dos dados abertos da própria
Câmara.

A pergunta é uma só: **de tudo que um partido apresenta, quanto é de cada tema?**
De cada 100 propostas do PSOL, quantas são de Direitos Humanos; do NOVO, quantas
são de Finanças Públicas. O denominador é sempre o próprio partido, e é isso que
torna as bancadas comparáveis entre si — uma bancada de 3.690 projetos e outra de
128 aparecem na mesma escala, porque o que se compara é a composição da pauta, não
o volume.

**Fonte:** [Dados Abertos da Câmara dos Deputados](https://dadosabertos.camara.leg.br/) —
os arquivos anuais `proposicoes`, `proposicoesTemas` e `proposicoesAutores`, de
2023 a 2026, baixados de
`https://dadosabertos.camara.leg.br/arquivos/{assunto}/csv/{assunto}-{ano}.csv` e
guardados numa pasta `dados/` ao lado deste notebook. São ~365 MB, por isso não
vão versionados.

---

## 1. Bibliotecas

`pandas` para as tabelas, `glob` para achar os arquivos anuais e `os` para tirar o
ano do nome de cada arquivo.

In [1]:
import pandas as pd

In [2]:
import glob

In [3]:
import os

## 2. Carregar as três bases

A Câmara publica um arquivo por ano e por assunto. São três recortes do mesmo
universo, ligados por duas chaves diferentes:

| base | uma linha é | chave |
|---|---|---|
| `proposicoes` | uma proposição (PL, PEC, requerimento...) | `id` |
| `proposicoesTemas` | uma classificação temática de uma proposição | `uriProposicao` |
| `proposicoesAutores` | uma assinatura de autoria | `idProposicao` |

O padrão se repete três vezes: `glob` lista os arquivos anuais, o laço lê um a um,
anota de qual ano veio (`ano_arquivo`, que serve para auditoria e não entra na
análise) e `concat` empilha tudo num dataframe só.

Os caminhos são relativos: o notebook roda em qualquer máquina, desde que aberto
a partir da pasta onde ele está e com a `dados/` preenchida.

### Proposições

In [4]:
arquivos_proposicoes = glob.glob('dados/proposicoes-*.csv')

In [5]:
proposicoes = []

In [6]:
for arquivo in arquivos_proposicoes:
    df = pd.read_csv(arquivo, sep=';', low_memory=False)
    df['ano_arquivo'] = int(os.path.basename(arquivo).split('-')[1][:4])
    proposicoes.append(df)

proposicoes = pd.concat(proposicoes, ignore_index=True)

### Temas

`proposicoesTemas` chama a chave de `uriProposicao`; renomear para `uri` já na
carga é o que permite o merge mais adiante.

In [7]:
arquivos_temas = glob.glob('dados/proposicoesTemas-*.csv')

In [8]:
temas = []

In [9]:
for arquivo_tema in arquivos_temas:
    df = pd.read_csv(arquivo_tema, sep=';', low_memory=False)
    df['ano_arquivo'] = int(os.path.basename(arquivo_tema).split('-')[1][:4])
    df.rename(columns={'uriProposicao':'uri'}, inplace=True)
    temas.append(df)

temas = pd.concat(temas, ignore_index=True)

### Autores

Mesma coisa do outro lado: `idProposicao` vira `id`. Uma linha por assinatura —
projetos com dezenas de apoiadores aparecem dezenas de vezes aqui.

In [10]:
arquivos_autores = glob.glob('dados/proposicoesAutores-*.csv')

In [11]:
autores = []

In [12]:
for arquivo_autores in arquivos_autores:
    df = pd.read_csv(arquivo_autores, sep=';', low_memory=False)
    df['ano_arquivo'] = int(os.path.basename(arquivo_autores).split('-')[1][:4])
    df.rename(columns={'idProposicao':'id'}, inplace=True)
    autores.append(df)

autores = pd.concat(autores, ignore_index=True)

## 3. Juntar as três bases

Depois dos dois merges, **uma linha é a combinação (assinatura × proposição ×
tema)**. Isso multiplica bastante: uma proposição com 3 temas e 40 assinaturas
vira 120 linhas. Os recortes das próximas seções desfazem essa multiplicação.

In [13]:
autores_propostas_temas = autores.merge(proposicoes, on='id').merge(temas, on='uri')

In [14]:
autores_propostas_temas.columns

Index(['id', 'uriProposicao', 'idDeputadoAutor', 'uriAutor', 'codTipoAutor',
       'tipoAutor', 'nomeAutor', 'siglaPartidoAutor', 'uriPartidoAutor',
       'siglaUFAutor', 'ordemAssinatura', 'proponente', 'ano_arquivo_x', 'uri',
       'siglaTipo_x', 'numero_x', 'ano_x', 'codTipo', 'descricaoTipo',
       'ementa', 'ementaDetalhada', 'keywords', 'dataApresentacao',
       'uriOrgaoNumerador', 'uriPropAnterior', 'uriPropPrincipal',
       'uriPropPosterior', 'urlInteiroTeor', 'urnFinal',
       'ultimoStatus_dataHora', 'ultimoStatus_sequencia',
       'ultimoStatus_uriRelator', 'ultimoStatus_idOrgao',
       'ultimoStatus_siglaOrgao', 'ultimoStatus_uriOrgao',
       'ultimoStatus_regime', 'ultimoStatus_descricaoTramitacao',
       'ultimoStatus_idTipoTramitacao', 'ultimoStatus_descricaoSituacao',
       'ultimoStatus_idSituacao', 'ultimoStatus_despacho',
       'ultimoStatus_apreciacao', 'ultimoStatus_url', 'ano_arquivo_y',
       'siglaTipo_y', 'numero_y', 'ano_y', 'codTema', 'tema', 

Das dezenas de colunas das três bases, ficam as catorze que interessam: quem
assinou, em que ordem, por qual partido, que tipo de proposição é, quando foi
apresentada e qual o tema.

O sufixo `_x` vem do merge — `proposicoes` e `temas` têm colunas de mesmo nome
(`siglaTipo`, `numero`, `ano`) e o pandas desambigua com `_x` / `_y`.

In [15]:
autores_propostas_temas_limpo = autores_propostas_temas[['id', 'uri','nomeAutor','tipoAutor','ordemAssinatura', 'proponente', 'siglaPartidoAutor', 'siglaTipo_x', 'numero_x', 'ano_x',  'dataApresentacao', 'ultimoStatus_descricaoTramitacao', 'ultimoStatus_descricaoSituacao', 'tema']]

In [16]:
autores_propostas_temas_limpo

,id,uri,nomeAutor,tipoAutor,ordemAssinatura,proponente,siglaPartidoAutor,siglaTipo_x,numero_x,ano_x,dataApresentacao,ultimoStatus_descricaoTramitacao,ultimoStatus_descricaoSituacao,tema
0,2642434,https://dadosabertos.camara.leg.br/api/v2/prop...,Senado Federal - Senadora Damares Alves,Órgão do Poder Legislativo,1,1,REPUBLIC,PL,5099,2023,2026-08-10T15:14:20,Publicação de Documento,NaN,Direitos Humanos e Minorias
1,2642434,https://dadosabertos.camara.leg.br/api/v2/prop...,Senado Federal - Senadora Damares Alves,Órgão do Poder Legislativo,1,1,REPUBLIC,PL,5099,2023,2026-08-10T15:14:20,Publicação de Documento,NaN,Saúde
2,2638818,https://dadosabertos.camara.leg.br/api/v2/prop...,Senado Federal - Confúcio Moura,Órgão do Poder Legislativo,1,1,MDB,PL,5926,2023,2026-07-09T17:36:00,Apresentação de Proposição,Aguardando Despacho do Presidente da Câmara do...,Trabalho e Emprego
3,2638818,https://dadosabertos.camara.leg.br/api/v2/prop...,Senado Federal - Confúcio Moura,Órgão do Poder Legislativo,1,1,MDB,PL,5926,2023,2026-07-09T17:36:00,Apresentação de Proposição,Aguardando Despacho do Presidente da Câmara do...,Finanças Públicas e Orçamento
4,2638805,https://dadosabertos.camara.leg.br/api/v2/prop...,Senado Federal - Flávio Arns,Órgão do Poder Legislativo,1,1,PSD,PL,786,2023,2026-07-09T16:57:00,Apresentação de Proposição,Aguardando Despacho do Presidente da Câmara do...,Comunicações
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176177,2599885,https://dadosabertos.camara.leg.br/api/v2/prop...,Poder Executivo,Órgão do Poder Executivo,1,1,NaN,PL,1,2026,2026-01-02T10:11:00,Desapensação,Arquivada,Finanças Públicas e Orçamento
176178,2168236,https://dadosabertos.camara.leg.br/api/v2/prop...,Laura Carneiro,Deputado(a),1,1,PMDB,PL,1242,2026,2018-02-21T15:43:00,Notificações,Aguardando Despacho do Presidente da Câmara do...,Direito Civil e Processual Civil
176179,2168236,https://dadosabertos.camara.leg.br/api/v2/prop...,Laura Carneiro,Deputado(a),1,1,PMDB,PL,1242,2026,2018-02-21T15:43:00,Notificações,Aguardando Despacho do Presidente da Câmara do...,Direitos Humanos e Minorias
176180,2168236,https://dadosabertos.camara.leg.br/api/v2/prop...,Carmen Zanotto,Deputado(a),2,1,PPS,PL,1242,2026,2018-02-21T15:43:00,Notificações,Aguardando Despacho do Presidente da Câmara do...,Direito Civil e Processual Civil


## 4. Primeiro recorte: só projetos de lei

`PL` e `PLP` são projeto de lei ordinária e complementar. Ficam de fora
requerimentos, indicações, PECs e recursos, que são instrumentos de tramitação e
não proposta de política pública. Requerimento é o tipo mais numeroso da base:
sem esse corte, a análise viraria uma medida de atividade parlamentar, não de
pauta.

In [17]:
autores_propostas_temas_limpo = autores_propostas_temas_limpo[(autores_propostas_temas_limpo['siglaTipo_x'] == 'PL') | (autores_propostas_temas_limpo['siglaTipo_x'] == 'PLP')]

In [18]:
autores_propostas_temas_limpo['siglaTipo_x'].unique()

array(['PL', 'PLP'], dtype=object)

## 5. A pauta de cada bancada

Esta célula faz três coisas.

**Consolida as siglas.** A base escreve a mesma sigla de várias formas (`REP`,
`REPUBLIC`, `REPUBLICA`; `SOLIDARI` pelo SOLIDARIEDADE) e mantém siglas extintas
de partidos que se fundiram — DEM e PSL viraram UNIÃO, PPS virou CIDADANIA, PMDB
virou MDB. Sem consolidar, a mesma bancada aparece repartida em várias linhas, a
maioria com dois ou três projetos.

**Aplica os outros dois recortes.**

- `tipoAutor == 'Deputado(a)'` exclui comissões, o Executivo e órgãos externos. O
  caso menos óbvio é o Senado: projetos que chegam de lá trazem sigla de partido e
  seriam creditados à bancada da Câmara.
- `ordemAssinatura == 1` deixa só o autor principal. A unidade da análise é o
  projeto, não a assinatura: um PL com 40 apoiadores conta uma vez, para quem o
  apresentou.

**Conta.** `partidos_temas` fica com uma linha por (partido, tema) e a coluna
`total_tema`, que é quantas propostas daquela bancada foram classificadas naquele
tema.

In [19]:
SIGLAS = {'REP':'REPUBLICANOS','REPUBLIC':'REPUBLICANOS','REPUBLICA':'REPUBLICANOS',
          'PODE':'PODEMOS', 'PMDB':'MDB', 'PPS':'CIDADANIA', 'DEM':'UNIÃO', 'PSL':'UNIÃO',
          'SOLIDARI':'SOLIDARIEDADE'}

base = autores_propostas_temas_limpo[
    (autores_propostas_temas_limpo['tipoAutor'] == 'Deputado(a)') &
    (autores_propostas_temas_limpo['ordemAssinatura'] == 1)
].copy()

base['partido'] = (base['siglaPartidoAutor'].str.upper().str.strip()
                   .replace(SIGLAS))

partidos_temas = (base.groupby('partido')['tema']
                  .value_counts().reset_index(name='total_tema'))

## 6. A fatia de cada tema na pauta

`total_geral` repete, em cada linha, o total do partido — é o que `transform('sum')`
faz: devolve o resultado do grupo no formato da tabela original, sem precisar de um
segundo merge. `% da pauta` é a divisão de um pelo outro.

Somando as linhas de um partido, dá 100%: cada bancada distribui a própria pauta
entre os temas, e é essa distribuição que a análise compara.

> **Atenção ao denominador.** `total_geral` soma **pares (projeto, tema)**, não
> projetos. A base de temas repete a proposição uma vez por tema, e cada PL tem 2,3
> temas em média — por isso o total do PL dá 8.529, e não os 3.690 projetos que ele
> de fato apresentou. Para a coluna `% da pauta` isso está correto, porque
> numerador e denominador contam a mesma coisa e as fatias fecham em 100%; mas para
> dizer "a bancada apresentou N projetos" é preciso contar `id` único.

In [20]:
partidos_temas['total_geral'] = (
    partidos_temas.groupby('partido')['total_tema'].transform('sum'))

partidos_temas['% da pauta'] = (
    partidos_temas['total_tema'] / partidos_temas['total_geral'] * 100).round(2)

O resultado é a tabela que alimenta os gráficos: partido, tema, quantas propostas
e que fatia da pauta daquela bancada isso representa.

In [21]:
partidos_temas

,partido,tema,total_tema,total_geral,% da pauta
0,AVANTE,Direitos Humanos e Minorias,80,458,17.47
1,AVANTE,Saúde,38,458,8.30
2,AVANTE,Trabalho e Emprego,36,458,7.86
3,AVANTE,Administração Pública,33,458,7.21
4,AVANTE,Direito Penal e Processual Penal,32,458,6.99
...,...,...,...,...,...
662,UNIÃO,Relações Internacionais e Comércio Exterior,34,5115,0.66
663,UNIÃO,Turismo,29,5115,0.57
664,UNIÃO,Estrutura Fundiária,25,5115,0.49
665,UNIÃO,Processo Legislativo e Atuação Parlamentar,13,5115,0.25


---

## O gráfico

Desta tabela sai um waffle de 100 células para cada uma das 21 maiores bancadas:
cada célula é 1% da pauta do partido, pintada pelo tema. Os 14 temas de maior
interesse levam cor; os outros 18 ficam num bloco cinza no fim, que é o que mantém
cada bloco fechando em 100% da pauta — uma célula de Saúde é 1% de tudo o que
aquela bancada apresentou, não 1% do recorte colorido.

O script que desenha o gráfico não faz parte deste repositório: ele consome
exatamente a coluna `% da pauta` calculada acima.

## Ressalvas

- **2026 é um ano parcial.** Os arquivos vão até a data do download (agosto de
  2026), então o último ano da legislatura tem menos projetos que os demais. Como
  todas as leituras são proporcionais dentro da própria bancada, isso não distorce
  a comparação entre partidos — mas inviabiliza qualquer série temporal.
- **Tema é classificação da Câmara**, atribuída pelo serviço de documentação da
  Casa, e uma proposição pode ter mais de um. Não é uma leitura do conteúdo do
  projeto.
- **A análise mede o que foi apresentado, não o que foi aprovado.** Apresentar
  projeto é barato; nada aqui diz respeito a tramitação ou resultado.
- **Autoria é do primeiro assinante.** Coautoria e apoiamento não entram.
- **Partido é o do momento do registro** na base de autores, não a filiação atual
  do parlamentar. Trocas de partido no meio da legislatura ficam onde estavam.
- **Bancadas pequenas oscilam.** Num partido com 21 propostas no total, um único
  projeto vale cinco pontos percentuais. Os gráficos cortam em 100 propostas por
  isso.